In [0]:
# Importação das bibliotecas necessárias

from dotenv import load_dotenv
from pathlib import Path
import os

from pyspark.sql import functions as F

In [0]:
# Localiza e carrega o arquivo .env com as configurações

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

tabela_destino = "squad2.ecommerce_clientes"

print(f"Tabela de análise: {tabela_destino}")

In [0]:
# Lê a tabela de clientes diretamente do SQL Server

df_clientes_analise = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .load()

print("Tabela lida do SQL Server com sucesso.")

In [0]:
# Validação da estrutura e amostragem dos dados

print("=== ESTRUTURA DOS DADOS ===")
df_clientes_analise.printSchema()

print("\n=== AMOSTRA DOS DADOS ===")
display(df_clientes_analise.limit(10))

In [0]:
# Validação do volume de dados

quantidade_linhas = df_clientes_analise.count()
quantidade_colunas = len(df_clientes_analise.columns)

print("=== VOLUME DE DADOS ===")
print(f"Linhas   : {quantidade_linhas}")
print(f"Colunas  : {quantidade_colunas}")

In [0]:
# Validação da chave primária id_cliente

nulos_id_cliente = df_clientes_analise.filter(
    F.col("id_cliente").isNull()
).count()

print("=== INTEGRIDADE DA CHAVE PRIMÁRIA ===")
print(f"Registros com id_cliente nulo: {nulos_id_cliente}")

if nulos_id_cliente == 0:
    print("Status: saudável - nenhum registro com id_cliente nulo.")
else:
    print("Status: atenção - existem registros com id_cliente nulo.")